In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(q1_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df=df.drop(columns=['Order_ID'])
df.head()


In [ ]:
# Task 2: Write your code here:
print(df.isnull().sum())
for col in df.columns[df.isnull().any()]:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())



In [ ]:
# Task 3: Write your code here:
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

feature_cols = [col for col in df.columns if col != 'Delivery_Time']
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [ ]:
# Task 6: Write your code here:
import seaborn as sns

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
y = df['Delivery_Time']
X = df.drop(columns=['Delivery_Time'])




In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
fold_models = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)
    fold_models.append(rf_model)

    print(f"Fold {fold}: MAE = {mae:.4f}")


In [ ]:
# Task 1: Write your code here:
feature_importance = np.mean([model.feature_importances_ for model in fold_models], axis=0)
importance_df = pd.DataFrame({'feature': X.columns, 'importance': feature_importance}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
all_predictions = fold_models[-1].predict(X)
plt.figure(figsize=(8, 5))
plt.hist(all_predictions, bins=30, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.title('Predicted Delivery Time Distribution')
plt.show()


In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_val)

    cb_model = CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_val)

    ensemble_pred = (rf_pred + cb_pred) / 2
    ensemble_mae = mean_absolute_error(y_val, ensemble_pred)
    ensemble_mae_scores.append(ensemble_mae)

    print(f"Fold {fold}: Ensemble MAE = {ensemble_mae:.4f}")

print(f"\nAverage Ensemble MAE: {np.mean(ensemble_mae_scores):.4f}")